# 01 Data loading and sanity checks
Convert the 1.3 GB Kaggle CSV into a compact Parquet file once, draw a 100k-row development sample, and check that the data is what we expect before any analysis.

**Before running:** select the "Python (ctds)" kernel (see the README).

In [19]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # project root, so `src` can be imported

import pandas as pd
from IPython.display import display

from src.config import RAW_CSV, FLIGHTS_PARQUET, TABLES_DIR
from src.data_loading import csv_to_parquet, make_sample, load_flights, compare_dtype_memory
from src.profiling import track, MB

FORCE = False  # True = rebuild the Parquet file and the sample

pd.set_option("display.float_format", "{:,.3f}".format)
print("CSV:    ", RAW_CSV, "| exists:", RAW_CSV.exists())
print("Parquet:", FLIGHTS_PARQUET, "| exists:", FLIGHTS_PARQUET.exists())

CSV:     D:\ComputationalTools\data\raw\flight_data_2024.csv | exists: True
Parquet: D:\ComputationalTools\data\processed\flights_2024.parquet | exists: True


## 1. CSV → Parquet (chunked)

We read the CSV in chunks of 500,000 rows. We keep only the columns we need (`usecols`) and give them compact dtypes: `category` for codes, `int8` for flags and calendar fields, `float32` for minutes and miles. The leaking actual-time columns (`taxi_out`, `wheels_off`, `air_time`, ...) are **never read**. No rows are dropped at this stage.

In [20]:
conversion_stats = csv_to_parquet(force=FORCE)
conversion_stats

flights_2024.parquet already exists -> skipping (use force=True to redo).


### Why optimized dtypes matter

We compare the memory footprint on the first 200,000 rows. The ratio extrapolates to the full ~7M rows, and the table goes in the report.

In [21]:
if RAW_CSV.exists():
    dtype_table = compare_dtype_memory(nrows=200_000)
    dtype_table.to_csv(TABLES_DIR / "01_dtype_memory.csv", index=False)
    display(dtype_table)
else:
    print("Raw CSV not found -> skipping the dtype comparison.")

,setup,columns,memory_mb_for_200000_rows
0,"all 35 columns, default dtypes",35,65.000
1,"our columns, default dtypes",23,41.600
2,"our columns, optimized dtypes",23,12.000


## 2. Development sample (100,000 rows)

In [22]:
sample_stats = make_sample(n=100_000, force=FORCE)
sample_stats

flights_2024_sample.parquet already exists -> skipping (use force=True to redo).


## 3. Load and inspect

In [23]:
path = FLIGHTS_PARQUET
with track("load_parquet", source=path.name) as s:
    df = load_flights(path)
    s["rows"] = len(df)

print(f"Loaded {len(df):,} rows x {df.shape[1]} columns from {path.name}")
print(f"Memory in pandas: {df.memory_usage(deep=True).sum() / MB:,.1f} MB")
df.dtypes.to_frame("dtype").T

[load_parquet] runtime: 0.69 s | peak memory (Python/pandas): 94.7 MB | peak memory (pyarrow pool): 1035.6 MB
Loaded 7,079,081 rows x 23 columns from flights_2024.parquet
Memory in pandas: 425.4 MB


,fl_date,op_unique_carrier,origin,dest,origin_state_nm,dest_state_nm,cancellation_code,month,day_of_month,day_of_week,...,crs_arr_time,crs_elapsed_time,distance,dep_delay,arr_delay,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay
dtype,datetime64[us],category,category,category,category,category,category,int8,int8,int8,...,Int16,float32,float32,float32,float32,float32,float32,float32,float32,float32


In [24]:
df.head()

,fl_date,op_unique_carrier,origin,dest,origin_state_nm,dest_state_nm,cancellation_code,month,day_of_month,day_of_week,...,crs_arr_time,crs_elapsed_time,distance,dep_delay,arr_delay,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay
0,2024-01-01,9E,JFK,DTW,New York,Michigan,NaN,1,1,1,...,1508,136.000,509.000,-5.000,-19.000,0.000,0.000,0.000,0.000,0.000
1,2024-01-01,9E,MSP,CLE,Minnesota,Ohio,NaN,1,1,1,...,1325,130.000,622.000,-14.000,-30.000,0.000,0.000,0.000,0.000,0.000
2,2024-01-01,9E,JFK,RIC,New York,Virginia,NaN,1,1,1,...,1601,106.000,288.000,-4.000,-20.000,0.000,0.000,0.000,0.000,0.000
3,2024-01-01,9E,RIC,JFK,Virginia,New York,NaN,1,1,1,...,1841,111.000,288.000,-7.000,-42.000,0.000,0.000,0.000,0.000,0.000
4,2024-01-01,9E,DTW,MKE,Michigan,Wisconsin,NaN,1,1,1,...,1034,79.000,237.000,-5.000,-14.000,0.000,0.000,0.000,0.000,0.000


## 4. Sanity checks

### 4.1 Row count and time coverage

In [25]:
checks = {}  # saved as one table at the end

checks["rows"] = len(df)
checks["first_date"] = df["fl_date"].min().date()
checks["last_date"] = df["fl_date"].max().date()
checks["months_present"] = df["month"].nunique()
print(f"{checks['rows']:,} flights from {checks['first_date']} to {checks['last_date']}")
df["month"].value_counts().sort_index().to_frame("flights").T

7,079,081 flights from 2024-01-01 to 2024-12-31


month,1,2,3,4,5,6,7,8,9,10,11,12
flights,547271,519221,591767,582205,609743,611132,634613,619025,582622,615497,575404,590581


### 4.2 Missing values per column

In [26]:
missing = pd.DataFrame({
    "n_missing": df.isna().sum(),
    "share_missing": df.isna().mean(),
}).sort_values("n_missing", ascending=False)
missing

,n_missing,share_missing
cancellation_code,6982766,0.986
arr_delay,113814,0.016
dep_delay,92970,0.013
crs_elapsed_time,1,0.000
origin,0,0.000
op_unique_carrier,0,0.000
fl_date,0,0.000
dest_state_nm,0,0.000
origin_state_nm,0,0.000
dest,0,0.000


### 4.3 Cancelled and diverted flights

Ground rule 3: these flights have no `arr_delay`. We give every flight an explicit `status` so that later steps can choose deliberately what to include.

In [27]:
status = pd.Series("completed", index=df.index)
status[df["diverted"] == 1] = "diverted"
status[df["cancelled"] == 1] = "cancelled"
df["status"] = status.astype("category")

checks["share_cancelled"] = (df["status"] == "cancelled").mean()
checks["share_diverted"] = (df["status"] == "diverted").mean()
display(df["status"].value_counts(normalize=True).to_frame("share"))

# Cancellation causes: A = carrier, B = weather, C = National Air System, D = security
display(df.loc[df["status"] == "cancelled", "cancellation_code"]
          .value_counts(normalize=True).to_frame("share of cancellations"))

,share
status,
completed,0.984
cancelled,0.014
diverted,0.002


,share of cancellations
cancellation_code,
B,0.557
A,0.321
C,0.122
D,0.000


In [28]:
arr_missing_by_status = df.groupby("status", observed=True)["arr_delay"].apply(lambda s: s.isna().mean())
display(arr_missing_by_status.to_frame("share arr_delay missing"))

checks["completed_without_arr_delay"] = int(df.loc[df["status"] == "completed", "arr_delay"].isna().sum())
print("Completed flights without arr_delay:", checks["completed_without_arr_delay"], "(expected: 0 or very few)")

,share arr_delay missing
status,
cancelled,1.000
completed,0.000
diverted,1.000


Completed flights without arr_delay: 0 (expected: 0 or very few)


### 4.4 Arrival delays (completed flights only)

In [29]:
arr = df.loc[df["status"] == "completed", "arr_delay"].dropna()

checks["share_arr_delay_15plus"] = (arr >= 15).mean()
checks["median_arr_delay"] = arr.median()
checks["mean_arr_delay"] = arr.mean()
checks["max_arr_delay"] = arr.max()
print(f"Share of completed flights arriving 15+ min late: {checks['share_arr_delay_15plus']:.1%}")
print("(The proposal's 10k-row sample gave 21.5%.)")

arr.quantile([0, 0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 0.999, 1]).to_frame("arr_delay (min)").T

Share of completed flights arriving 15+ min late: 20.8%
(The proposal's 10k-row sample gave 21.5%.)


,0.000,0.010,0.050,0.250,0.500,0.750,0.950,0.990,0.999,1.000
arr_delay (min),-126.000,-37.000,-27.000,-15.000,-6.000,9.000,83.000,215.000,753.000,"3,803.000"


### 4.5 Airports, carriers, routes

In [30]:
checks["n_origin_airports"] = df["origin"].nunique()
checks["n_airports_total"] = len(set(df["origin"].unique()) | set(df["dest"].unique()))
checks["n_carriers"] = df["op_unique_carrier"].nunique()
checks["n_routes_directed"] = df.groupby(["origin", "dest"], observed=True).ngroups
checks["n_route_carrier_pairs"] = df.groupby(["origin", "dest", "op_unique_carrier"], observed=True).ngroups

for k in ["n_origin_airports", "n_airports_total", "n_carriers", "n_routes_directed", "n_route_carrier_pairs"]:
    print(f"{k:24s} {checks[k]:>8,}")

n_origin_airports             348
n_airports_total              348
n_carriers                     15
n_routes_directed           6,805
n_route_carrier_pairs      13,258


In [31]:
# Decides how many route+carrier pairs have enough flights for a reliable RQ1 profile.
per_pair = df.groupby(["origin", "dest", "op_unique_carrier"], observed=True).size()
per_pair.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).to_frame("flights per route+carrier").T

,count,mean,std,min,10%,25%,50%,75%,90%,max
flights per route+carrier,"13,258.000",533.948,688.850,1.000,13.000,81.000,311.500,716.000,"1,366.600","7,622.000"


### 4.6 Leakage check: do the cause columns sum to `arr_delay`?

The proposal found this on a 10k-row sample. Here we verify it on the full data. If it holds, the cause columns encode the target and must stay out of all features.

In [32]:
cause_cols = ["carrier_delay", "weather_delay", "nas_delay", "security_delay", "late_aircraft_delay"]
late = df[(df["status"] == "completed") & (df["arr_delay"] >= 15)]

cause_sum = late[cause_cols].sum(axis=1, min_count=1)
checks["late_flights_with_causes"] = late[cause_cols].notna().all(axis=1).mean()
checks["causes_sum_to_arr_delay"] = (cause_sum - late["arr_delay"]).abs().lt(0.5).mean()
checks["not_late_with_all_causes_zero"] = df.loc[df["arr_delay"] < 15, cause_cols].eq(0).all(axis=1).mean()

print(f"15+ late flights with all cause columns filled: {checks['late_flights_with_causes']:.2%}")
print(f"... of which causes sum to arr_delay:           {checks['causes_sum_to_arr_delay']:.2%}")
print(f"Flights < 15 min late with all causes equal to 0: {checks['not_late_with_all_causes_zero']:.2%}")

15+ late flights with all cause columns filled: 100.00%
... of which causes sum to arr_delay:           100.00%
Flights < 15 min late with all causes equal to 0: 100.00%


### 4.7 Schedule columns (what RQ1 will cluster on)

These columns are known before departure. We check their ranges so we know what cleaning the feature step needs. For example, BTS writes midnight as `2400`.

In [17]:
sched = df[["crs_dep_time", "crs_arr_time", "crs_elapsed_time", "distance"]]
display(sched.describe().T)

checks["crs_dep_time_2400"] = int((df["crs_dep_time"] == 2400).sum())
checks["crs_elapsed_nonpositive_or_missing"] = int((~(df["crs_elapsed_time"] > 0)).sum())
checks["distance_nonpositive_or_missing"] = int((~(df["distance"] > 0)).sum())
print("crs_dep_time == 2400:", checks["crs_dep_time_2400"])
print("crs_elapsed_time <= 0 or missing:", checks["crs_elapsed_nonpositive_or_missing"])
print("distance <= 0 or missing:", checks["distance_nonpositive_or_missing"])

,count,mean,std,min,25%,50%,75%,max
crs_dep_time,"7,079,081.000","1,327.300",493.031,1.000,905.000,"1,320.000","1,736.000","2,400.000"
crs_arr_time,"7,079,081.000","1,491.365",519.438,1.000,"1,103.000","1,517.000","1,925.000","2,359.000"
crs_elapsed_time,"7,079,080.000",146.766,72.387,-160.000,93.000,130.000,177.000,"1,326.000"
distance,"7,079,081.000",833.906,596.254,11.000,399.000,680.000,"1,069.000","5,095.000"


crs_dep_time == 2400: 1
crs_elapsed_time <= 0 or missing: 2
distance <= 0 or missing: 0


## 5. Save the summary

In [18]:
summary = pd.Series(checks, name="value").to_frame()
summary["source"] = path.name
summary.to_csv(TABLES_DIR / "01_sanity_checks.csv")
summary

,value,source
rows,7079081,flights_2024.parquet
first_date,2024-01-01,flights_2024.parquet
last_date,2024-12-31,flights_2024.parquet
months_present,12,flights_2024.parquet
share_cancelled,0.014,flights_2024.parquet
share_diverted,0.002,flights_2024.parquet
completed_without_arr_delay,0,flights_2024.parquet
share_arr_delay_15plus,0.208,flights_2024.parquet
median_arr_delay,-6.000,flights_2024.parquet
mean_arr_delay,7.098,flights_2024.parquet


In [ ]:
#TEST EVERYTHING (WE CAN DELETE LATER)

import pandas as pd
from src.config import RAW_CSV, FLIGHTS_PARQUET
from src.data_loading import load_flights

cols = ["arr_delay", "dep_delay", "distance", "cancelled", "diverted"]

# Totals computed straight from the CSV (read in chunks, default pandas types)
n_csv, sums_csv = 0, pd.Series(0.0, index=cols)
for chunk in pd.read_csv(RAW_CSV, usecols=cols, chunksize=500_000):
    n_csv += len(chunk)
    sums_csv += chunk.sum()

# The same totals from the Parquet file
pq = load_flights(FLIGHTS_PARQUET, columns=cols)
sums_pq = pq.astype("float64").sum()

print("rows:", n_csv, "vs", len(pq))
pd.DataFrame({"csv": sums_csv, "parquet": sums_pq, "diff": sums_csv - sums_pq})

rows: 7079081 vs 7079081


,csv,parquet,diff
arr_delay,"49,441,174.000","49,441,174.000",0.000
dep_delay,"88,563,502.000","88,563,502.000",0.000
distance,"5,903,289,185.000","5,903,289,185.000",0.000
cancelled,"96,315.000","96,315.000",0.000
diverted,"17,499.000","17,499.000",0.000
